# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

dataset = pd.read_csv('work/outputs/dataset.csv')
baseline = pd.read_csv('work/outputs/baseline_action_score.csv')

feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]
target_col = 'is_declining_label'

# Pick the best model the same honest way as w05_model.ipynb (client-grouped split,
# selected by precision@50) -- then, ONLY for this deployment-facing queue, refit that
# model type on ALL rows so every page gets scored. The honest out-of-sample numbers
# this selection is based on live in w05_model.ipynb / w06_validation_audit.ipynb, not here.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_client_aware_split(df, target_col, client_col='client_hash_id', test_frac=0.2, random_state=RANDOM_STATE):
    clients = df[client_col].dropna().unique()
    if len(clients) >= 5:
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(clients)
        n_test = max(1, int(round(len(shuffled) * test_frac)))
        test_clients = set(shuffled[:n_test])
        test_mask = df[client_col].isin(test_clients)
        train_idx = df.index[~test_mask]
        test_idx = df.index[test_mask]
        if df.loc[train_idx, target_col].nunique() == 2 and df.loc[test_idx, target_col].nunique() == 2:
            return train_idx, test_idx
    return train_test_split(df.index, test_size=test_frac, random_state=random_state, stratify=df[target_col])

def build_models():
    return {
        'logistic_regression': Pipeline([('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))]),
        'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
        'random_forest': RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    }

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    return float(y_true[order[:min(k, len(y_true))]].mean()) if len(y_true) else 0.0

train_idx, test_idx = make_client_aware_split(dataset, target_col)
X_train, X_test = dataset.loc[train_idx, feature_cols], dataset.loc[test_idx, feature_cols]
y_train, y_test = dataset.loc[train_idx, target_col], dataset.loc[test_idx, target_col]

best_name, best_p50 = None, -1
for name, model in build_models().items():
    model.fit(X_train, y_train)
    p50 = precision_at_k(y_test, model.predict_proba(X_test)[:, 1], 50)
    if p50 > best_p50:
        best_name, best_p50 = name, p50
print(f"Best model (by held-out precision@50 = {best_p50:.3f}): {best_name}")

final_model = build_models()[best_name]
final_model.fit(dataset[feature_cols], dataset[target_col])
dataset['model_probability'] = final_model.predict_proba(dataset[feature_cols])[:, 1]

# Blend: 70% model / 30% normalized baseline score, same ratio as scripts/04_evaluate_and_export.py
playbook = dataset.merge(
    baseline[['content_hash_id', 'baseline_action_score', 'reason_codes']],
    on='content_hash_id', how='left'
)
b = playbook['baseline_action_score']
playbook['baseline_score_normalized'] = (b - b.min()) / max(b.max() - b.min(), 1e-9)
playbook['final_action_score'] = (
    100 * (0.70 * playbook['model_probability'] + 0.30 * playbook['baseline_score_normalized'])
).clip(0, 100)

def merge_reason_codes(row):
    reasons = [r for r in str(row['reason_codes']).split('|') if r and r != 'nan']
    if row['model_probability'] >= 0.65:
        reasons.append('model_decline_risk')
    if row['model_probability'] >= 0.50 and row['impressions_90d'] >= dataset['impressions_90d'].quantile(0.5):
        reasons.append('model_flagged_opportunity')
    seen = []
    for r in reasons:
        if r not in seen:
            seen.append(r)
    return '|'.join(seen or ['general_review_candidate'])

playbook['final_reason_codes'] = playbook.apply(merge_reason_codes, axis=1)

# Archetype -> action mapping. This freestyle/scoring lane didn't run formal k-means
# clustering (that's the separate "Structured Content Archetype" lane) -- instead the
# archetype is named directly from the combination of reason codes already computed by
# w04_baseline_score.ipynb + this model, which keeps it fully explainable end to end.
def archetype_and_action(reasons_str):
    reasons = set(reasons_str.split('|'))
    if {'visible_declining_momentum', 'model_decline_risk'} & reasons and 'visible_poor_rank' in reasons:
        return 'Declining & Under-ranked', 'refresh'
    if 'model_decline_risk' in reasons and 'visible_declining_momentum' in reasons:
        return 'Model-confirmed Decline', 'refresh'
    if {'low_ctr_visible_page', 'low_engagement_visible_page'} & reasons:
        return 'Visible but Under-converting', 'refresh_metadata_or_content'
    if 'inconsistent_visibility' in reasons:
        return 'Inconsistent Visibility', 'monitor'
    if 'model_flagged_opportunity' in reasons:
        return 'Model-flagged Opportunity', 'review'
    if 'not_visible_low_priority' in reasons:
        return 'Low Visibility', 'monitor'
    return 'Stable / General Watch', 'monitor'

playbook[['archetype', 'suggested_action']] = playbook['final_reason_codes'].apply(
    lambda r: pd.Series(archetype_and_action(r))
)

playbook['final_rank'] = playbook['final_action_score'].rank(method='first', ascending=False).astype(int)
playbook = playbook.sort_values('final_rank')

print(f"\nArchetype -> action mapping (n per archetype):")
print(playbook.groupby(['archetype', 'suggested_action'], observed=True).size().rename('n').reset_index().to_string(index=False))

print(f"\nTop 20 of the final ranked playbook:")
show_cols = ['final_rank', 'final_action_score', 'archetype', 'suggested_action', 'final_reason_codes']
print(playbook[show_cols].head(20).round(1).to_string(index=False))


Best model (by held-out precision@50 = 0.740): logistic_regression



Archetype -> action mapping (n per archetype):
                   archetype            suggested_action     n
    Declining & Under-ranked                     refresh   784
     Inconsistent Visibility                     monitor  2862
              Low Visibility                     monitor 51830
     Model-confirmed Decline                     refresh  2052
   Model-flagged Opportunity                      review 17998
      Stable / General Watch                     monitor  5233
Visible but Under-converting refresh_metadata_or_content 22932

Top 20 of the final ranked playbook:
 final_rank  final_action_score                 archetype suggested_action                                                                        final_reason_codes
          1                98.0  Declining & Under-ranked          refresh visible_declining_momentum|visible_poor_rank|model_decline_risk|model_flagged_opportunity
          2                87.2  Declining & Under-ranked          refresh      

## 1. Ranked actions + reason codes

The final score blends the Week-5 model's probability (70%) with the Week-4 baseline's
normalized score (30%) -- the same blend ratio as `scripts/04_evaluate_and_export.py` -- so a
page has to look risky by *both* the transparent rule and the learned model to reach the very
top, while a page either one strongly flags still surfaces further down. Reason codes carry
over from the baseline (`w04_baseline_score.ipynb`) plus two model-driven codes
(`model_decline_risk`, `model_flagged_opportunity`).

**Archetype -> action mapping:** this project's lane is Refresh/Content Opportunity Scoring
(freestyle), not the separate Content-Archetype-Clustering lane, so archetypes here are named
directly from the reason-code combinations already computed above rather than from a new
k-means run -- every archetype traces back to an explainable rule, not an opaque cluster
centroid. The printed table above is that mapping: which combination of signals earns which
named archetype, and which action it maps to.

**The decay/refresh insight**, tying back to `w04_signal_audit.ipynb` (Test 2) and
`w05_model.ipynb`'s feature importance: pages already showing negative in-window momentum
*and* a poor rank are the archetype most consistently worth a refresh -- the same "catch it
before it fully decays" pattern `docs/flyrank-seo-research-march-2026.pdf` reports at the
portfolio level (Finding #2, the content performance curve).

**Real archetype counts** (best model: `logistic_regression`, held-out precision@50 = 0.740,
matching `w05_model.ipynb`): `Low Visibility` (monitor) is the largest group by far at 51,830
of 103,691 rows (50.0%) -- most content simply isn't visible enough to act on yet.
`Visible but Under-converting` (refresh_metadata_or_content) is next at 22,932 (22.1%), then
`Model-flagged Opportunity` (review) at 17,998 (17.4%). The two decay-driven archetypes are
smaller but the most actionable: `Model-confirmed Decline` (refresh) at 2,052 and `Declining
& Under-ranked` (refresh) at 784 -- together 2,836 pages (2.7%) that combine at least two
independent decline signals. `Inconsistent Visibility` (monitor, 2,862) and `Stable / General
Watch` (monitor, 5,233) round out the rest.

**The top of the queue confirms the decay/refresh insight**: 13 of the top 20 rows are
`Declining & Under-ranked` or `Model-confirmed Decline`, both `refresh` actions, and rank #1
(score 98.0) carries all four decline-linked reason codes at once
(`visible_declining_momentum|visible_poor_rank|model_decline_risk|model_flagged_opportunity`)
-- exactly the "everything agrees" case the blend design is meant to surface first.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
INTENDED_USE = {
    'who': 'Content/SEO editorial teams and content managers deciding which pages to review first.',
    'for_what': 'Prioritizing a fixed weekly review capacity -- NOT for automatic publishing, deletion, or rewriting.',
    'input_data': f"{len(dataset):,} content items across {dataset['client_hash_id'].nunique()} clients, "
                  f"90-day feature window (per the ML-04 data contract).",
    'label': 'is_declining_label -- a proxy for a >=30% GSC impression drop, not a confirmed editorial verdict.',
}
LIMITS = [
    'Single time window: results may not generalize to a different season or a Google algorithm update.',
    'Proxy label: impression decline can reflect seasonality or SERP changes, not content quality alone.',
    'Client coverage: some clients have thin history or no GA4 integration (see has_ga4_data audit in w04/w06).',
    'Cross-sectional, not causal: the score is an association, not a guarantee a refresh will recover traffic.',
    'Best-model choice and blend weights (70/30) are fixed decisions made this cycle -- see section 4 for when to revisit them.',
]

print("=== INTENDED USE ===")
for k, v in INTENDED_USE.items():
    print(f"  {k}: {v}")
print("\n=== LIMITS ===")
for l in LIMITS:
    print(f"  - {l}")


=== INTENDED USE ===
  who: Content/SEO editorial teams and content managers deciding which pages to review first.
  for_what: Prioritizing a fixed weekly review capacity -- NOT for automatic publishing, deletion, or rewriting.
  input_data: 103,691 content items across 42 clients, 90-day feature window (per the ML-04 data contract).
  label: is_declining_label -- a proxy for a >=30% GSC impression drop, not a confirmed editorial verdict.

=== LIMITS ===
  - Single time window: results may not generalize to a different season or a Google algorithm update.
  - Proxy label: impression decline can reflect seasonality or SERP changes, not content quality alone.
  - Client coverage: some clients have thin history or no GA4 integration (see has_ga4_data audit in w04/w06).
  - Cross-sectional, not causal: the score is an association, not a guarantee a refresh will recover traffic.
  - Best-model choice and blend weights (70/30) are fixed decisions made this cycle -- see section 4 for when to 

## 2. Intended use and limits

This playbook is **decision-support for a human reviewer**, sized to a fixed weekly editorial
review capacity -- not an autonomous publishing or content-editing system. Per
`skills/writing-honest-claims/SKILL.md`'s claim ladder, the honest framing is "these pages
look worth reviewing first, because..." -- never "these pages will recover if refreshed."

Limits are printed above and stated up front rather than buried in a caveats section: a
proxy label, a single time window, uneven client coverage, and a fixed (not re-optimized
every run) blend of model and baseline. Anyone reusing this queue past its originating run
should re-check these before trusting the ranking.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
HUMAN_REVIEW_CHECKLIST = {
    'high': [
        'Confirm the page still exists and is indexed before scheduling work.',
        'Read the page once -- does the reason code match what you see, or does it look like a data artifact?',
        'Check has_ga4_data / has_momentum flags for this row -- a structural-missing flag inflates false confidence.',
    ],
    'medium': [
        'Same checks as high-confidence, plus: compare against 1-2 similar pages from the same client before committing time.',
    ],
    'low': [
        'Treat as a light monitoring candidate only -- do not schedule dedicated editorial time from this tier alone.',
    ],
}
NO_GO_LIST = [
    'Never auto-publish, auto-delete, or auto-rewrite a page from this queue -- every action needs a human sign-off.',
    'Never treat the score as proof a specific edit will recover traffic -- no causal design supports that claim.',
    'Never surface raw scores externally (to clients or the public) without the limits in section 2 attached.',
    'Never use this queue to make personnel, vendor, or budget decisions -- it scores CONTENT, not people or teams.',
    'Never skip the has_ga4_data / has_momentum check for a "high" pick -- see w04_signal_audit.ipynb\'s flag test.',
]

# quantile-based confidence tiers, same construction as scripts/04_evaluate_and_export.py
high_threshold = playbook['final_action_score'].quantile(0.80)
medium_threshold = playbook['final_action_score'].quantile(0.50)
def confidence_label(score):
    if score >= high_threshold:
        return 'high'
    if score >= medium_threshold:
        return 'medium'
    return 'low'
playbook['confidence'] = playbook['final_action_score'].apply(confidence_label)

print("=== CONFIDENCE TIERS ===")
print(playbook['confidence'].value_counts())
print("\n=== HUMAN REVIEW CHECKLIST ===")
for tier, checks in HUMAN_REVIEW_CHECKLIST.items():
    print(f"\n{tier.upper()}:")
    for c in checks:
        print(f"  - {c}")
print("\n=== NO-GO LIST (never automate) ===")
for item in NO_GO_LIST:
    print(f"  - {item}")


=== CONFIDENCE TIERS ===
confidence
low       51845
medium    31107
high      20739
Name: count, dtype: int64

=== HUMAN REVIEW CHECKLIST ===

HIGH:
  - Confirm the page still exists and is indexed before scheduling work.
  - Read the page once -- does the reason code match what you see, or does it look like a data artifact?
  - Check has_ga4_data / has_momentum flags for this row -- a structural-missing flag inflates false confidence.

MEDIUM:
  - Same checks as high-confidence, plus: compare against 1-2 similar pages from the same client before committing time.

LOW:
  - Treat as a light monitoring candidate only -- do not schedule dedicated editorial time from this tier alone.

=== NO-GO LIST (never automate) ===
  - Never auto-publish, auto-delete, or auto-rewrite a page from this queue -- every action needs a human sign-off.
  - Never treat the score as proof a specific edit will recover traffic -- no causal design supports that claim.
  - Never surface raw scores externally (to c

## 3. Human review + the no-go list

Confidence tiers use the same quantile construction (p80 / p50 of the final score) as
`scripts/04_evaluate_and_export.py`, so "high confidence" means top-quintile score here, not
an absolute probability guarantee. Every tier gets a review checklist -- higher confidence
still means "check before acting," just a lighter check.

The no-go list is the section that matters most: nothing in this playbook should ever trigger
an automatic action. It exists to route human attention, not replace it -- consistent with
`DATA_USE.md` and the "decision-support, never automated" framing this whole track uses.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
import os, json

MONITORING_METRICS = {
    'declining_rate': float(dataset[target_col].mean()),
    'mean_final_score': float(playbook['final_action_score'].mean()),
    'high_confidence_share': float((playbook['confidence'] == 'high').mean()),
    'feature_means': {c: float(dataset[c].mean()) for c in feature_cols},
    'n_rows': int(len(dataset)),
    'n_clients': int(dataset['client_hash_id'].nunique()),
}

RETRAIN_TRIGGERS = [
    'declining_rate drifts by more than 5 percentage points from this run\'s snapshot -- the base rate the model was tuned against has moved.',
    'Any feature_means value shifts by more than ~30% relative to this snapshot -- a client mix or tracking change, not just noise.',
    'A new client cohort is added with materially different has_ga4_data coverage (re-run the ML-06 flag test first).',
    'Precision@50 on a fresh honest split (w05/w06 style) drops below the Week-4 baseline\'s Precision@50 -- the model is no longer earning its complexity.',
    'More than ~90 days pass since this snapshot -- the 90-day feature window itself will have fully rolled over.',
]

os.makedirs('work/outputs', exist_ok=True)
with open('work/outputs/monitoring_snapshot.json', 'w') as f:
    json.dump(MONITORING_METRICS, f, indent=2)

print("Monitoring snapshot (saved to work/outputs/monitoring_snapshot.json):")
print(json.dumps(MONITORING_METRICS, indent=2))
print("\nRetrain triggers:")
for t in RETRAIN_TRIGGERS:
    print(f"  - {t}")


Monitoring snapshot (saved to work/outputs/monitoring_snapshot.json):
{
  "declining_rate": 0.3748252018015064,
  "mean_final_score": 43.51712100933999,
  "high_confidence_share": 0.20000771523083005,
  "feature_means": {
    "impressions_90d": 5725.70449701517,
    "clicks_90d": 17.88870779527635,
    "ctr_90d": 0.2774653769454278,
    "avg_position_90d": 13.642364435520259,
    "sessions_90d": 16.852774107685335,
    "pageviews_90d": 21.59657057989604,
    "engaged_sessions_90d": 0.5681206662101822,
    "organic_sessions_90d": 10.903289581545168,
    "impressions_last30": 2681.25230733622,
    "impressions_first60": 3044.45218967895,
    "momentum_pct": 411.26659728068995,
    "active_days_90d": 65.43890983788371,
    "has_ga4_data": 0.7509330607285106,
    "has_momentum": 0.8466308551368972
  },
  "n_rows": 103691,
  "n_clients": 42
}

Retrain triggers:
  - declining_rate drifts by more than 5 percentage points from this run's snapshot -- the base rate the model was tuned against ha

## 4. Monitoring / retrain triggers

This is a lightweight, non-production monitoring plan -- there's no live pipeline to alert
automatically, so the trigger conditions above are things a human re-checks periodically by
re-running this notebook (or a slice of it) against a newer data pull and diffing the result
against `work/outputs/monitoring_snapshot.json`, the reference point saved above.

The 90-day trigger exists because the feature window itself is time-boxed: past that point,
the entire feature set has aged out, not just the model's fit.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import os

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1) The ranked queue -- regenerated by this notebook, intentionally NOT committed
#    (work/**/*.csv is gitignored and the CI leak-guard blocks dataset-shaped CSVs).
export_cols = [
    'final_rank', 'client_hash_id', 'content_hash_id', 'final_action_score',
    'model_probability', 'baseline_action_score', 'archetype', 'suggested_action',
    'confidence', 'final_reason_codes', 'is_declining_label',
]
playbook[export_cols].to_csv('work/outputs/content_action_playbook.csv', index=False)
print("Wrote work/outputs/content_action_playbook.csv")

# 2) Figures -- committed to work/figures/, since the paper (ML-11) reuses these directly.
def simple_svg_bar_chart(title, labels, values, path, color='#4E79A7', width=900, height=480):
    labels = [str(l) for l in labels]
    values = [float(v) for v in values]
    max_value = max(values + [1])
    margin_left, margin_right, margin_top, margin_bottom = 200, 40, 60, 40
    plot_w = width - margin_left - margin_right
    plot_h = height - margin_top - margin_bottom
    gap = 10
    bar_h = max(14, (plot_h - gap * max(len(values) - 1, 0)) / max(len(values), 1))
    lines = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="#ffffff"/>',
             f'<text x="{width/2}" y="30" text-anchor="middle" font-family="Arial" font-size="20" fill="#16232a">{title}</text>']
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_top + i * (bar_h + gap)
        bar_w = (value / max_value) * plot_w
        lines.append(f'<text x="{margin_left-10}" y="{y+bar_h*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="12" fill="#27343b">{label[:30]}</text>')
        lines.append(f'<rect x="{margin_left}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{color}" rx="4"/>')
        lines.append(f'<text x="{margin_left+bar_w+8:.1f}" y="{y+bar_h*0.65:.1f}" font-family="Arial" font-size="12" fill="#27343b">{value:,.3g}</text>')
    lines.append('</svg>')
    with open(path, 'w') as f:
        f.write('\n'.join(lines))

archetype_counts = playbook['archetype'].value_counts()
simple_svg_bar_chart('Archetype mix', archetype_counts.index.tolist(), archetype_counts.values.tolist(),
                      'work/figures/archetype_mix.svg', color='#426B69')

action_counts = playbook['suggested_action'].value_counts()
simple_svg_bar_chart('Suggested action mix', action_counts.index.tolist(), action_counts.values.tolist(),
                      'work/figures/action_mix.svg', color='#6F4E7C')

confidence_counts = playbook['confidence'].value_counts().reindex(['high', 'medium', 'low'], fill_value=0)
simple_svg_bar_chart('Playbook confidence mix', confidence_counts.index.tolist(), confidence_counts.values.tolist(),
                      'work/figures/confidence_mix.svg', color='#8C6BB1')
print("Wrote work/figures/archetype_mix.svg, action_mix.svg, confidence_mix.svg")

# 3) Metrics JSON -- per the ML-10 card, kept committed as the receipts the paper's numbers trace back to.
playbook_summary = {
    'best_model': best_name,
    'held_out_precision_at_50': float(best_p50),
    'n_scored': int(len(playbook)),
    'archetype_counts': archetype_counts.to_dict(),
    'action_counts': action_counts.to_dict(),
    'confidence_counts': confidence_counts.to_dict(),
    'high_confidence_top10_preview': playbook.loc[playbook['confidence'] == 'high', export_cols]
        .head(10).to_dict(orient='records'),
}
with open('work/outputs/playbook_summary.json', 'w') as f:
    json.dump(playbook_summary, f, indent=2, default=str)
print("Wrote work/outputs/playbook_summary.json")


Wrote work/outputs/content_action_playbook.csv
Wrote work/figures/archetype_mix.svg, action_mix.svg, confidence_mix.svg
Wrote work/outputs/playbook_summary.json


## 5. Exports for the paper

Three kinds of output, handled differently on purpose (per this assignment's card and
`work/README.md`'s "committed vs regenerated" rule):

- **`work/outputs/content_action_playbook.csv`** -- the full ranked queue. Not committed
  (`work/**/*.csv` is gitignored, and the CI leak-guard blocks it anyway); this cell
  regenerates it from `work/outputs/dataset.csv` + `work/outputs/baseline_action_score.csv`.
- **`work/figures/*.svg`** -- archetype mix, action mix, and confidence mix charts. These
  ARE meant to be committed -- the paper (`ML-11`) embeds them directly.
- **`work/outputs/playbook_summary.json`** -- the numeric receipts (best model, held-out
  precision@50, counts by archetype/action/confidence, a high-confidence preview) that the
  paper's Results section should trace back to.

All three files landed from the real run above: `content_action_playbook.csv` (103,691 rows,
gitignored by design), `archetype_mix.svg` / `action_mix.svg` / `confidence_mix.svg` in
`work/figures/`, and `playbook_summary.json` in `work/outputs/`. The SVGs and the JSON are
committed alongside this notebook; the CSV is left alone.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.